In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

def compute_vix_index(call_prices, put_prices, call_strikes, put_strikes, expiration_date, r, S, K):
    # Compute the time to expiration
    current_date = datetime.strptime(expiration_date, "%Y-%m-%d")
    expiration_date = datetime.strptime(expiration_date, "%Y-%m-%d")
    T = (expiration_date - current_date).days / 365.0
    
    # Find the nearest call and put strikes to the forward price
    forward_price = S * np.exp(r * T)
    nearest_call_index = np.argmin(np.abs(call_strikes - forward_price))
    nearest_put_index = np.argmin(np.abs(put_strikes - forward_price))

    # Compute the midpoint of the nearest call and put prices
    call_price = call_prices[nearest_call_index]
    put_price = put_prices[nearest_put_index]
    midpoint = (call_price + put_price) / 2.0

    # Compute the VIX index
    VIX = 100.0 * np.sqrt(T) * midpoint / K

    return VIX

# Define a function to compute implied volatility using the Newton-Raphson method
def implied_volatility(option_price, S, K, T, r, option_type):
    # Define the required constants
    MAX_ITERATIONS = 100
    PRECISION = 1.0e-5

    # Initialize the variables
    sigma = 0.5
    for i in range(MAX_ITERATIONS):
        # Calculate the option price using the Black-Scholes formula
        price = black_scholes(option_type, S, K, T, r, sigma)

        # Calculate the difference between the model price and the market price
        diff = price - option_price

        # Check if the difference is within the required precision
        if abs(diff) < PRECISION:
            return sigma

        # Calculate the Vega (rate of change of option price with respect to volatility)
        vega = black_scholes_vega(S, K, T, r, sigma)

        # Update the volatility estimate using the Newton-Raphson method
        sigma = sigma - diff / vega

    # If the iterations exceed the maximum limit, return None
    return None

# Define a function to compute the Black-Scholes option price
def black_scholes(option_type, S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == 'put':
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    return price

# Define a function to compute the Vega (rate of change of option price with respect to volatility)
def black_scholes_vega(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T)

    return vega

# Define the directory and file paths
directory = '/Users/Farid Srouji/Documents/Research/master/Data/2013-01/'
start_date = pd.Timestamp('2013-01-01')
end_date = pd.Timestamp('2013-01-31')

# Iterate over each date
current_date = start_date
while current_date <= end_date:
    option_file_path = os.path.join(directory, current_date.strftime("%Y-%m-%d") + "options.cvs")
    stock_file_path = os.path.join(directory, current_date.strftime("%Y-%m-%d") + "stocks.cvs")

    # Check if the option file exists
    if not os.path.exists(option_file_path):
        print(f"Option file not found for date {current_date}. Skipping...")
        current_date += pd.DateOffset(days=1)
        continue

    # Check if the stock file exists
    if not os.path.exists(stock_file_path):
        print(f"Stock file not found for date {current_date}. Skipping...")
        current_date += pd.DateOffset(days=1)
        continue

    # Load the options data
    options_data = pd.read_csv(option_file_path, header=0)
    options_data.columns = ['contract', 'underlying', 'expiration', 'type', 'strike', 'style', 'bid', 'bid_size',
                            'ask', 'ask_size', 'volume', 'open_interest', 'quote_date', 'delta', 'gamma',
                            'theta', 'vega', 'implied_volatility']

    # Load the stock data
    stock_data = pd.read_csv(stock_file_path, header=0)
    stock_data.columns = ['symbol', 'open', 'high', 'low', 'close', 'volume', 'adjust_close']

    # Merge the options and stock data based on the common stock symbol
    merged_data = pd.merge(options_data, stock_data, left_on='underlying', right_on='symbol')

    # Group the merged data by stock
    stock_groups = merged_data.groupby('symbol')

    # Iterate over each stock group
    for stock, group in stock_groups:
        # Extract the necessary data for the VIX computation
        call_prices = group['ask'].values
        put_prices = group['ask'].values
        call_strikes = group['strike'].values
        put_strikes = group['strike'].values
        T = group['expiration'].values
        r = 0.05  # Assuming a risk-free rate of 5%
        S = group['close'].values
        K = group['close'].values

        # Compute the VIX index for the current stock
        vix_index = compute_vix_index(call_prices, put_prices, call_strikes, put_strikes, T, r, S, K)

        # Print or store the VIX timeseries for the current stock
        print(f"VIX timeseries for stock {stock}:")
        print(vix_index)
        print("------------------------")

    # Move to the next date
    current_date += pd.DateOffset(days=1)
